# Digital Cash Management Forecast: daily flows viewer

Reads the Parquet files of `daily_flows` (what staging writes and every model gets: one row per client, day and
category with `money_in`, `money_out` and `postings`) and shows them. It only reads: nothing here changes the
project. Set `FOLDER` to `load_test_flows` to look at the load test's fake data instead.

**Kernel:** the project's code env (the setup's `CODE_ENV`, which has pyarrow), like `forecast_lab.ipynb`.

In [ ]:
FOLDER = "daily_flows"       # the folder to read: daily_flows (the real data) or load_test_flows (the load test's)
SUMMARY_FILES = 10           # how many Parquet files the summary reads (-1: all of them; slower on the full data)
SAMPLE_ROWS = 20             # rows shown from the first file
CLIENT_NO = ""               # a client number to show day by day (empty: the first client of the first file)

In [ ]:
# Reads only: nothing in this notebook changes the project.
import io
import time

import dataiku
import pandas as pd
import pyarrow.parquet as pq

from cashflow import CURRENT, _daily_files, _plain

folder, paths = _daily_files(FOLDER)
if not paths:
    raise RuntimeError(f"No Parquet file in the folder {FOLDER}: run staging (or the scenario Load test) first")
with folder.get_download_stream(CURRENT) as stream:
    version = stream.read().decode().strip()


def read_file(path, columns=None, filters=None):
    """One Parquet file of the folder as a DataFrame."""
    with folder.get_download_stream(path) as stream:
        data = stream.read()
    return _plain(pq.read_table(io.BytesIO(data), columns=columns, filters=filters).to_pandas())


print(f"{FOLDER}: current version {version}, {len(paths)} Parquet files")

## 1. Files

Each Parquet file of the current version and its size.

In [ ]:
# Each Parquet file of the current version and its size (no file is read here).
files = pd.DataFrame({"file": paths, "size_mb": [round(folder.get_path_details(p).get("size", 0) / 1e6, 2)
                                                 for p in paths]})
print(f"{len(files)} files, {files['size_mb'].sum():,.1f} MB in all")
display(files)

## 2. Summary

Rows, clients, postings and money per category, over the first `SUMMARY_FILES` files.

In [ ]:
# Totals over the first SUMMARY_FILES files (each client's rows are all in one file), read one file at a time.
chosen = paths if SUMMARY_FILES in (-1, None) else paths[:SUMMARY_FILES]
started, parts, clients_read = time.time(), [], 0
for i, path in enumerate(chosen, 1):
    d = read_file(path)
    d["net"] = d["money_in"] - d["money_out"]
    clients_read += d["client_no"].nunique()
    parts.append(d.groupby("category").agg(rows=("client_no", "size"), clients=("client_no", "nunique"),
                                           postings=("postings", "sum"), money_in=("money_in", "sum"),
                                           money_out=("money_out", "sum"), net=("net", "sum"),
                                           first_day=("day", "min"), last_day=("day", "max")))
    if i % 10 == 0 or i == len(chosen):
        print(f"read {i} of {len(chosen)} files ({time.time() - started:.0f} s)")
by_category = pd.concat(parts).groupby(level=0).agg({"rows": "sum", "clients": "sum", "postings": "sum",
                                                     "money_in": "sum", "money_out": "sum", "net": "sum",
                                                     "first_day": "min", "last_day": "max"})
print(f"{'All' if len(chosen) == len(paths) else f'First {len(chosen)} of {len(paths)}'} files: "
      f"{by_category['rows'].sum():,} rows, {clients_read:,} clients, {by_category['postings'].sum():,} postings, "
      f"from {by_category['first_day'].min():%Y-%m-%d} to {by_category['last_day'].max():%Y-%m-%d}")
display(by_category.round(0).sort_values("rows", ascending=False))

## 3. Sample rows

The columns and the first rows of the first file.

In [ ]:
first = read_file(paths[0])
print(f"{paths[0]}: {len(first):,} rows, {first['client_no'].nunique():,} clients. Columns:")
print(first.dtypes.to_string())
display(first.head(SAMPLE_ROWS))

## 4. One client

`CLIENT_NO` day by day, and per month.

In [ ]:
# One client's daily rows and its net cash flow per month (money in, money out, net). Its rows are all in one file:
# the files are read until it is found.
wanted = str(CLIENT_NO).strip() or first["client_no"].iloc[0]
rows = pd.DataFrame()
for path in paths:
    rows = read_file(path, filters=[("client_no", "=", wanted)])
    if len(rows):
        break
if rows.empty:
    print(f"Client {wanted} is not in {FOLDER}")
else:
    rows = rows.sort_values(["day", "category"]).reset_index(drop=True)
    print(f"Client {wanted}: {len(rows):,} rows, {rows['day'].min():%Y-%m-%d} to {rows['day'].max():%Y-%m-%d}, "
          f"{rows['postings'].sum():,} postings, in {path}")
    display(rows.tail(SAMPLE_ROWS))
    monthly = (rows.assign(month=rows["day"].dt.to_period("M").dt.to_timestamp())
               .groupby("month")[["money_in", "money_out"]].sum())
    monthly["net"] = monthly["money_in"] - monthly["money_out"]
    display(monthly.round(0))
    ax = monthly.plot(figsize=(12, 4), title=f"Client {wanted}: money in, money out and net per month (SAR)")
    ax.axhline(0, color="grey", linewidth=0.8)